
# GVH Diagonal Cubic 0.3.2.6 — Full Tensor vs Reduced Radial Consistency Audit

**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2.6  
**Partie :** `0.2C2_weak_field_predictions`  
**Statut :** consistency audit — candidate timelike/vector family

---

## Objective

Notebook 0.3.2.5 obtained explicit radial equations by inserting the areal-radius static spherical ansatz before varying the reduced action.

That produced the conditional weak-field result

\[
\operatorname{rank}J=3,
\]

and, after transformation to isotropic coordinates,

\[
\gamma=1,\qquad \beta=1
\]

for the aligned candidate branch, while the coupling

\[
c_{14}=c_1+c_4
\]

remained unfixed.

Before promoting those results to a more robust status, this notebook tests whether the radial reduction lost an independent metric equation.

We therefore compare two routes:

\[
\boxed{
\text{general spherical metric}
\rightarrow
\text{variation}
\rightarrow
C(r)=r^2
}
\]

against

\[
\boxed{
C(r)=r^2
\rightarrow
\text{reduced action}
\rightarrow
\text{variation}
}.
\]

The first route retains an independent angular metric function \(C(r)\) until after variation and is therefore a stronger consistency test than gauge-fixing the areal radius at the start.

**Important:** this is a gauge-unfixed spherical variational audit. It is a strong proxy for the full tensor check, but it is not claimed to replace a completely arbitrary-coordinate tensor-CAS variation of the four-dimensional action.



## 0. Scientific safeguards

The notebook must distinguish:

- `EXACT_GEOMETRIC` — directly computed geometric identity;
- `CANDIDATE_ACTION_CONDITIONAL` — result conditional on the action ansatz of 0.3.2.2;
- `CONSISTENCY_PASS` — two independent reduction routes agree;
- `ORDER_CONSISTENT` — an additional equation is satisfied through the order actually solved;
- `BLOCKED_FULL_CAS_VARIATION` — a fully general 4D variation has not yet been independently reproduced;
- `NOT_UNIQUE_GVH_PREDICTION` — \(c_{14}\) and the action-selection problem remain open.

No observational data enter this notebook.


In [1]:

from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import sympy as sp

NOTEBOOK_ID = "GVH_Diagonal_Cubic_0.3.2.6"
VERSION = "0.3.2.6"

print(NOTEBOOK_ID, VERSION)
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH_Diagonal_Cubic_0.3.2.6 0.3.2.6
Python: 3.12.13
SymPy: 1.14.0



# 1. Gauge-unfixed static spherical metric

Instead of imposing the areal-radius gauge immediately, use

\[
ds^2
=
-A(r)\,dt^2
+
B(r)\,dr^2
+
C(r)\left(
d\theta^2+\sin^2\theta\,d\phi^2
\right).
\]

The function \(C(r)\) is kept independent during variation.

Only after deriving the three metric Euler–Lagrange equations do we impose

\[
C(r)=r^2.
\]


In [2]:

t, r, th, ph = sp.symbols("t r theta phi", real=True)

A = sp.Function("A")(r)
B = sp.Function("B")(r)
Cang = sp.Function("C")(r)

coords = [t, r, th, ph]

g = sp.diag(
    -A,
    B,
    Cang,
    Cang * sp.sin(th)**2
)

g_inv = sp.simplify(g.inv())
n = 4

print("Metric determinant:")
sp.pprint(sp.factor(g.det()))


Metric determinant:
            2       2   
-A(r)⋅B(r)⋅C (r)⋅sin (θ)



# 2. Exact 4D geometry

Compute the Christoffel symbols, Ricci tensor and Ricci scalar directly from the gauge-unfixed metric.

This provides the geometric input independently of the equations stored in 0.3.2.5.


In [3]:

Gamma = [[[sp.Integer(0) for _ in range(n)] for _ in range(n)] for _ in range(n)]

for rho in range(n):
    for mu in range(n):
        for nu in range(n):
            expr = 0
            for sig in range(n):
                expr += g_inv[rho, sig] * (
                    sp.diff(g[sig, nu], coords[mu])
                    + sp.diff(g[sig, mu], coords[nu])
                    - sp.diff(g[mu, nu], coords[sig])
                )
            Gamma[rho][mu][nu] = sp.simplify(expr / 2)

Ricci = [[sp.Integer(0) for _ in range(n)] for _ in range(n)]

for mu in range(n):
    for nu in range(n):
        expr = 0
        for rho in range(n):
            expr += sp.diff(Gamma[rho][mu][nu], coords[rho])
            expr -= sp.diff(Gamma[rho][mu][rho], coords[nu])

            for sig in range(n):
                expr += Gamma[rho][rho][sig] * Gamma[sig][mu][nu]
                expr -= Gamma[rho][nu][sig] * Gamma[sig][mu][rho]

        Ricci[mu][nu] = sp.simplify(expr)

R_scalar = sp.simplify(
    sum(
        g_inv[mu,nu] * Ricci[mu][nu]
        for mu in range(n)
        for nu in range(n)
    )
)

print("Gauge-unfixed Ricci scalar derived.")
print("Expression length:", len(str(R_scalar)))


Gauge-unfixed Ricci scalar derived.
Expression length: 345



# 3. Aligned unit-timelike field

Use the same aligned branch as 0.3.2.5:

\[
u^\mu
=
\left(
A^{-1/2},0,0,0
\right).
\]

The normalization remains

\[
u^\mu u_\mu=-1.
\]

We recompute the vector invariants without imposing \(C=r^2\).


In [4]:

u_con = sp.Matrix([
    1/sp.sqrt(A),
    0,
    0,
    0
])

u_cov = sp.simplify(g * u_con)

norm = sp.simplify((u_con.T * u_cov)[0])

assert norm == -1

nabla_u_cov = [
    [sp.Integer(0) for _ in range(n)]
    for _ in range(n)
]

for mu in range(n):
    for nu in range(n):
        expr = sp.diff(u_cov[nu], coords[mu])

        for rho in range(n):
            expr -= Gamma[rho][mu][nu] * u_cov[rho]

        nabla_u_cov[mu][nu] = sp.simplify(expr)

print("u.u =", norm)


u.u = -1


In [5]:

# I1 = (nabla_mu u_nu)(nabla^mu u^nu)

I1 = 0

for mu in range(n):
    for nu in range(n):
        for aa in range(n):
            for bb in range(n):
                I1 += (
                    g_inv[mu,aa]
                    * g_inv[nu,bb]
                    * nabla_u_cov[mu][nu]
                    * nabla_u_cov[aa][bb]
                )

I1 = sp.simplify(I1)

# nabla_mu u^nu
nabla_u_con = [
    [sp.Integer(0) for _ in range(n)]
    for _ in range(n)
]

for mu in range(n):
    for nu in range(n):
        expr = sp.diff(u_con[nu], coords[mu])

        for rho in range(n):
            expr += Gamma[nu][mu][rho] * u_con[rho]

        nabla_u_con[mu][nu] = sp.simplify(expr)

theta_exp = sp.simplify(
    sum(nabla_u_con[mu][mu] for mu in range(n))
)

I3 = sp.simplify(
    sum(
        nabla_u_cov[mu][nu] * nabla_u_con[nu][mu]
        for mu in range(n)
        for nu in range(n)
    )
)

a_cov = sp.Matrix([
    sp.simplify(
        sum(
            u_con[nu] * nabla_u_cov[nu][mu]
            for nu in range(n)
        )
    )
    for mu in range(n)
])

a2 = sp.simplify(
    (a_cov.T * g_inv * a_cov)[0]
)

print("I1 =", I1)
print("theta =", theta_exp)
print("I3 =", I3)
print("a^2 =", a2)


I1 = -Derivative(A(r), r)**2/(4*A(r)**2*B(r))
theta = 0
I3 = 0
a^2 = Derivative(A(r), r)**2/(4*A(r)**2*B(r))



The aligned branch again gives

\[
I_1
=
-\frac{A'^2}{4A^2B},
\qquad
\nabla_\mu u^\mu=0,
\qquad
I_3=0,
\qquad
a_\mu a^\mu
=
\frac{A'^2}{4A^2B}.
\]

Thus the candidate vector Lagrangian still collapses to

\[
\boxed{
\mathcal L_u
=
c_{14}
\frac{A'^2}{4A^2B}
},
\qquad
c_{14}=c_1+c_4,
\]

even before the areal-radius gauge is imposed.


In [6]:

c14 = sp.symbols("c14", real=True)

L_u = sp.simplify(
    c14 * sp.diff(A,r)**2
    / (4*A**2*B)
)

expected_I1 = -sp.diff(A,r)**2/(4*A**2*B)
expected_a2 = sp.diff(A,r)**2/(4*A**2*B)

assert sp.simplify(I1 - expected_I1) == 0
assert theta_exp == 0
assert I3 == 0
assert sp.simplify(a2 - expected_a2) == 0

print("PASS — gauge-unfixed aligned vector reduction reproduces c14-only sector.")


PASS — gauge-unfixed aligned vector reduction reproduces c14-only sector.



# 4. Gauge-unfixed radial action

After integrating the angular dependence, the one-dimensional radial Lagrangian is proportional to

\[
L_{\rm rad}^{\rm unfixed}
=
C(r)\sqrt{A(r)B(r)}
\left[
R
+
c_{14}
\frac{A'^2}{4A^2B}
\right].
\]

We now vary independently with respect to

\[
A(r),\qquad B(r),\qquad C(r).
\]

This yields three reduced equations before choosing the areal-radius gauge.


In [7]:

L_rad_unfixed = sp.simplify(
    Cang * sp.sqrt(A*B) * (R_scalar + L_u)
)

def euler_lagrange_radial(L, field, max_order=2):
    result = sp.diff(L, field)

    for k in range(1, max_order + 1):
        derivative = sp.diff(field, r, k)
        term = sp.diff(L, derivative)

        if term != 0:
            result += (-1)**k * sp.diff(term, r, k)

    return sp.factor(sp.simplify(result))

E_A_unfixed = euler_lagrange_radial(
    L_rad_unfixed, A, max_order=2
)

E_B_unfixed = euler_lagrange_radial(
    L_rad_unfixed, B, max_order=2
)

E_C_unfixed = euler_lagrange_radial(
    L_rad_unfixed, Cang, max_order=2
)

print("Gauge-unfixed radial equations derived.")
print("Lengths:",
      len(str(E_A_unfixed)),
      len(str(E_B_unfixed)),
      len(str(E_C_unfixed)))


Gauge-unfixed radial equations derived.
Lengths: 429 189 432



# 5. Impose the areal-radius gauge only after variation

Now set

\[
C(r)=r^2,
\qquad
C'(r)=2r,
\qquad
C''(r)=2.
\]

This produces the three gauge-specialized equations

\[
E_A^{\rm unfixed\to areal}=0,
\quad
E_B^{\rm unfixed\to areal}=0,
\quad
E_C^{\rm unfixed\to areal}=0.
\]


In [8]:

areal_subs = {
    Cang: r**2,
    sp.diff(Cang,r): 2*r,
    sp.diff(Cang,r,2): 2,
}

E_A_areal_from_unfixed = sp.factor(
    sp.simplify(E_A_unfixed.subs(areal_subs))
)

E_B_areal_from_unfixed = sp.factor(
    sp.simplify(E_B_unfixed.subs(areal_subs))
)

E_C_areal_from_unfixed = sp.factor(
    sp.simplify(E_C_unfixed.subs(areal_subs))
)

print("A, B, angular equations specialized to C=r^2.")


A, B, angular equations specialized to C=r^2.



# 6. Reconstruct the 0.3.2.5 gauge-fixed equations independently

The gauge-fixed reduced equations obtained in 0.3.2.5 were

\[
E_A^{(0.3.2.5)}=0,
\]

\[
E_B^{(0.3.2.5)}=0.
\]

We reconstruct them here from their explicit formulas rather than importing numerical outputs.


In [9]:

E_A_old = sp.factor(
    -c14*r**2*A*B*sp.diff(A,r,2)/2
    + c14*r**2*A*sp.diff(A,r)*sp.diff(B,r)/4
    + 3*c14*r**2*B*sp.diff(A,r)**2/8
    - c14*r*A*B*sp.diff(A,r)
    + r*A**2*sp.diff(B,r)
    + A**2*B**2
    - A**2*B
)

E_B_old = sp.factor(
    -c14*r**2*sp.diff(A,r)**2/8
    - r*A*sp.diff(A,r)
    + A**2*B
    - A**2
)

print("0.3.2.5 equations reconstructed.")


0.3.2.5 equations reconstructed.



# 7. Direct consistency comparison

Two equations are physically equivalent if they differ only by a nonzero multiplicative prefactor on the nonsingular branch under study.

Compute

\[
\frac{
E_A^{\rm unfixed\to areal}
}{
E_A^{(0.3.2.5)}
},
\qquad
\frac{
E_B^{\rm unfixed\to areal}
}{
E_B^{(0.3.2.5)}
}.
\]


In [10]:

ratio_A = sp.factor(
    sp.simplify(
        E_A_areal_from_unfixed / E_A_old
    )
)

ratio_B = sp.factor(
    sp.simplify(
        E_B_areal_from_unfixed / E_B_old
    )
)

print("ratio_A =")
sp.pprint(ratio_A)

print("\nratio_B =")
sp.pprint(ratio_B)

assert sp.simplify(
    E_A_areal_from_unfixed - ratio_A*E_A_old
) == 0

assert sp.simplify(
    E_B_areal_from_unfixed - ratio_B*E_B_old
) == 0

print(
    "PASS — A and B equations from vary-before-gauge "
    "match the 0.3.2.5 gauge-fixed equations "
    "up to nonzero prefactors."
)


ratio_A =
  ___________
╲╱ A(r)⋅B(r) 
─────────────
  3     2    
 A (r)⋅B (r) 

ratio_B =
  ___________
╲╱ A(r)⋅B(r) 
─────────────
  2     2    
 A (r)⋅B (r) 
PASS — A and B equations from vary-before-gauge match the 0.3.2.5 gauge-fixed equations up to nonzero prefactors.



This is the main consistency result.

On the nonsingular branch \(A>0\), \(B>0\), the equations agree exactly up to the factors

\[
\frac{\sqrt{AB}}{A^3B^2}
\]

and

\[
\frac{\sqrt{AB}}{A^2B^2}.
\]

Therefore gauge-fixing \(C=r^2\) before variation did **not** alter the \(A\) and \(B\) equations obtained in 0.3.2.5.



# 8. The additional angular equation

The gauge-unfixed route also produces an independent-looking angular equation.

After setting \(C=r^2\), its numerator can be written as

\[
\begin{aligned}
E_{\Omega}={}&
c_{14} r B A'^2
-4rAB A''
+2rAA'B'
+2rB A'^2\\
&+4A^2B'
-4ABA'.
\end{aligned}
\]

The key question is whether this equation contradicts the weak-field solution of 0.3.2.5 at the order actually solved.


In [11]:

E_angular_num = sp.factor(
    c14*r*B*sp.diff(A,r)**2
    - 4*r*A*B*sp.diff(A,r,2)
    + 2*r*A*sp.diff(A,r)*sp.diff(B,r)
    + 2*r*B*sp.diff(A,r)**2
    + 4*A**2*sp.diff(B,r)
    - 4*A*B*sp.diff(A,r)
)

# Verify proportionality to the gauge-specialized C equation.
ratio_C = sp.factor(
    sp.simplify(
        E_C_areal_from_unfixed / E_angular_num
    )
)

assert sp.simplify(
    E_C_areal_from_unfixed
    - ratio_C*E_angular_num
) == 0

print("PASS — angular equation numerator identified.")
print("ratio_C =")
sp.pprint(ratio_C)


PASS — angular equation numerator identified.
ratio_C =
   ___________ 
 ╲╱ A(r)⋅B(r)  
───────────────
     2     2   
4⋅r⋅A (r)⋅B (r)



# 9. Weak-field audit of the angular equation

Reuse the 0.3.2.5 areal-radius expansion

\[
x=\frac{m}{r},
\]

\[
A
=
1-2a_1x+2a_2^{(R)}x^2+\cdots,
\]

\[
B
=
1+2b_1^{(R)}x+b_2^{(R)}x^2+\cdots.
\]

The previous solution was

\[
b_1^{(R)}=a_1,
\qquad
a_2^{(R)}=0,
\qquad
b_2^{(R)}
=
\frac{a_1^2}{2}(c_{14}+8).
\]

We substitute this solution into the angular equation and inspect the orders explicitly.


In [12]:

m, xeps = sp.symbols("m x", positive=True)

a1w, a2R, b1R, b2R = sp.symbols(
    "a1 a2R b1R b2R",
    real=True
)

x = m/r

A_series = (
    1
    - 2*a1w*x
    + 2*a2R*x**2
)

B_series = (
    1
    + 2*b1R*x
    + b2R*x**2
)

angular_series_raw = sp.expand(
    E_angular_num
    .subs({
        A: A_series,
        B: B_series,
    })
    .doit()
)

angular_x = sp.series(
    sp.simplify(
        angular_series_raw.subs(r, m/xeps)
    ),
    xeps,
    0,
    5
).removeO().expand()

solution_0325 = {
    b1R: a1w,
    a2R: 0,
    b2R: a1w**2*(c14+8)/2,
}

angular_on_solution = sp.factor(
    sp.expand(
        angular_x.subs(solution_0325)
    )
)

print("Angular weak-field series before solution:")
sp.pprint(sp.collect(angular_x, xeps))

print("\nAngular equation after 0.3.2.5 solution:")
sp.pprint(angular_on_solution)


Angular weak-field series before solution:
   ⎛    2                2                                                     ↪
 4 ⎜8⋅a₁ ⋅b1R⋅c₁₄   32⋅a₁ ⋅b1R   16⋅a₁⋅a2R⋅c₁₄   48⋅a₁⋅a2R   32⋅a₁⋅b2R   80⋅a2 ↪
x ⋅⎜───────────── - ────────── - ───────────── + ───────── + ───────── - ───── ↪
   ⎝      m             m              m             m           m           m ↪

↪      ⎞      ⎛    2           2                             ⎞                 ↪
↪ R⋅b1R⎟    3 ⎜4⋅a₁ ⋅c₁₄   8⋅a₁    40⋅a₁⋅b1R   32⋅a2R   8⋅b2R⎟    2 ⎛8⋅a₁   8⋅ ↪
↪ ─────⎟ + x ⋅⎜───────── - ───── + ───────── - ────── - ─────⎟ + x ⋅⎜──── - ── ↪
↪      ⎠      ⎝    m         m         m         m        m  ⎠      ⎝ m        ↪

↪     
↪ b1R⎞
↪ ───⎟
↪ m  ⎠

Angular equation after 0.3.2.5 solution:
     3  4          
24⋅a₁ ⋅x ⋅(c₁₄ + 4)
───────────────────
         m         



The angular equation vanishes through the coefficient orders used to determine

\[
b_1^{(R)},\qquad
a_2^{(R)},\qquad
b_2^{(R)}.
\]

Its first residual term appears at the next unsolved order:

\[
E_\Omega
\sim
\frac{24a_1^3}{m}
(c_{14}+4)x^4
+\cdots.
\]

This does **not** constitute an inconsistency of the second-order solution, because the metric ansatz was truncated before the third-order coefficients that can contribute at this order were introduced.

Therefore the appropriate status is:

`ORDER_CONSISTENT_THROUGH_SOLVED_WEAK_FIELD_ORDER`.


In [13]:

# The previous solution must eliminate the x^2 and x^3 coefficients.

angular_x2 = sp.simplify(
    angular_x.coeff(xeps,2)
    .subs(solution_0325)
)

angular_x3 = sp.simplify(
    angular_x.coeff(xeps,3)
    .subs(solution_0325)
)

assert angular_x2 == 0
assert angular_x3 == 0

print("PASS — angular equation vanishes through solved orders x^2 and x^3.")
print("x^2 residual:", angular_x2)
print("x^3 residual:", angular_x3)

print(
    "First displayed unsolved-order residual:",
    sp.factor(
        angular_x.coeff(xeps,4)
        .subs(solution_0325)
    )
)


PASS — angular equation vanishes through solved orders x^2 and x^3.
x^2 residual: 0
x^3 residual: 0
First displayed unsolved-order residual: 24*a1**3*(c14 + 4)/m



# 10. Does the rank-3 result survive?

The consistency audit does not change the three equations that generated the rank-3 result in 0.3.2.5.

Reconstruct the leading weak-field equations and recompute the Jacobian independently.


In [14]:

# Expand the reconstructed A/B radial equations.

EA_series_raw = sp.expand(
    E_A_old
    .subs({
        A: A_series,
        B: B_series,
    })
    .doit()
)

EB_series_raw = sp.expand(
    E_B_old
    .subs({
        A: A_series,
        B: B_series,
    })
    .doit()
)

EA_x = sp.series(
    sp.simplify(
        EA_series_raw.subs(r, m/xeps)
    ),
    xeps,
    0,
    4
).removeO().expand()

EB_x = sp.series(
    sp.simplify(
        EB_series_raw.subs(r, m/xeps)
    ),
    xeps,
    0,
    4
).removeO().expand()

eq_B1 = sp.expand(EB_x).coeff(xeps,1)
eq_A2 = sp.expand(EA_x).coeff(xeps,2)
eq_B2 = sp.expand(EB_x).coeff(xeps,2)

weak_unknowns = [
    b1R,
    a2R,
    b2R,
]

weak_eqs = [
    eq_B1,
    eq_A2,
    eq_B2,
]

J = sp.Matrix(weak_eqs).jacobian(
    weak_unknowns
)

J_on_solution = sp.simplify(
    J.subs(solution_0325)
)

rank_consistency = J_on_solution.rank()

print("Consistency-audited weak-field rank:", rank_consistency)
sp.pprint(J_on_solution)

assert rank_consistency == 3


Consistency-audited weak-field rank: 3
⎡     2          0     0 ⎤
⎢                        ⎥
⎢a₁⋅(8 - c₁₄)  -2⋅c₁₄  -1⎥
⎢                        ⎥
⎣   -8⋅a₁        4     1 ⎦



Thus

\[
\boxed{\operatorname{rank}J=3}
\]

survives the vary-before-gauge consistency audit.

The important refinement is:

> the rank-3 result is now supported by two independent spherical reduction routes, but it remains conditional on the candidate action, the aligned branch, and fixed \(a_1,c_{14}\).



# 11. Status of \(\gamma\) and \(\beta\)

0.3.2.5 already showed that after the necessary transformation to isotropic coordinates,

\[
\gamma=1,
\qquad
\beta=1
\]

for the aligned candidate branch.

Because the radial equations used in that derivation pass the present consistency audit, those conditional PPN-interface results are strengthened from

`REDUCED-ACTION-ONLY`

to

`SPHERICALLY-CONSISTENCY-AUDITED`.

They are still **not** promoted to unique GVH predictions because \(c_{14}\) and the action-selection problem remain open.


In [15]:

ppn_status = pd.DataFrame([
    {
        "interface": "gamma",
        "candidate_branch_value": 1,
        "status": "SPHERICALLY_CONSISTENCY_AUDITED",
        "unique_GVH_prediction": False,
    },
    {
        "interface": "beta",
        "candidate_branch_value": 1,
        "status": "SPHERICALLY_CONSISTENCY_AUDITED",
        "unique_GVH_prediction": False,
    },
])

ppn_status


,interface,candidate_branch_value,status,unique_GVH_prediction
0,gamma,1,SPHERICALLY_CONSISTENCY_AUDITED,False
1,beta,1,SPHERICALLY_CONSISTENCY_AUDITED,False



# 12. What this audit proves — and what it does not

## Strengthened results

The notebook establishes that:

1. keeping \(C(r)\) independent until after variation reproduces the 0.3.2.5 \(A\) and \(B\) equations exactly up to nonsingular multiplicative factors;
2. the additional angular equation does not contradict the previously solved weak-field orders;
3. the rank-3 coefficient closure survives;
4. therefore the second-order static spherical result is not an artifact of imposing \(C=r^2\) before variation.

## Still unresolved

The notebook does **not** establish:

1. a fully arbitrary-coordinate 4D tensor-CAS variation of \(T_{\mu\nu}^{(u)}\);
2. uniqueness of the candidate action;
3. a theoretical value of \(c_{14}\);
4. the role of \(c_2,c_3\) in non-aligned/time-dependent sectors;
5. ghost/stability viability;
6. preferred-frame PPN parameters;
7. a unique non-GR observational prediction.


In [16]:

consistency_summary = pd.DataFrame([
    {
        "test": "A equation: vary before gauge vs gauge-fixed reduction",
        "result": "PASS",
    },
    {
        "test": "B equation: vary before gauge vs gauge-fixed reduction",
        "result": "PASS",
    },
    {
        "test": "Angular equation through solved weak-field orders",
        "result": "PASS",
    },
    {
        "test": "Weak-field coefficient rank",
        "result": rank_consistency,
    },
    {
        "test": "Fully arbitrary-coordinate tensor variation",
        "result": "NOT_YET_INDEPENDENTLY_CAS_VERIFIED",
    },
    {
        "test": "Unique GVH prediction",
        "result": "BLOCKED",
    },
])

consistency_summary


,test,result
0,A equation: vary before gauge vs gauge-fixed r...,PASS
1,B equation: vary before gauge vs gauge-fixed r...,PASS
2,Angular equation through solved weak-field orders,PASS
3,Weak-field coefficient rank,3
4,Fully arbitrary-coordinate tensor variation,NOT_YET_INDEPENDENTLY_CAS_VERIFIED
5,Unique GVH prediction,BLOCKED



# 13. Final status logic


In [17]:

internal_checks = {
    "unit_normalization": norm == -1,
    "c14_reduction_reproduced": (
        sp.simplify(I1 - expected_I1) == 0
        and theta_exp == 0
        and I3 == 0
        and sp.simplify(a2 - expected_a2) == 0
    ),
    "A_equation_consistent": sp.simplify(
        E_A_areal_from_unfixed
        - ratio_A*E_A_old
    ) == 0,
    "B_equation_consistent": sp.simplify(
        E_B_areal_from_unfixed
        - ratio_B*E_B_old
    ) == 0,
    "angular_x2_consistent": angular_x2 == 0,
    "angular_x3_consistent": angular_x3 == 0,
    "rank3_survives": rank_consistency == 3,
    "no_observational_data_used": True,
    "unique_GVH_prediction_still_blocked": True,
}

for key, value in internal_checks.items():
    print(f"{key}: {value}")

assert all(internal_checks.values())

FINAL_STATUS = (
    "PASS-FULL-SPHERICAL-vs-REDUCED-RADIAL-CONSISTENCY_"
    "RANK-3-SURVIVES_PPN-GAMMA-BETA-STRENGTHENED_"
    "BLOCKED-FULL-4D-CAS-AND-UNIQUE-GVH-PREDICTION"
)

print("\nFINAL STATUS:", FINAL_STATUS)


unit_normalization: True
c14_reduction_reproduced: True
A_equation_consistent: True
B_equation_consistent: True
angular_x2_consistent: True
angular_x3_consistent: True
rank3_survives: True
no_observational_data_used: True
unique_GVH_prediction_still_blocked: True

FINAL STATUS: PASS-FULL-SPHERICAL-vs-REDUCED-RADIAL-CONSISTENCY_RANK-3-SURVIVES_PPN-GAMMA-BETA-STRENGTHENED_BLOCKED-FULL-4D-CAS-AND-UNIQUE-GVH-PREDICTION



# 14. Machine-readable audit artifact


In [18]:

artifact = {
    "notebook": NOTEBOOK_ID,
    "version": VERSION,
    "final_status": FINAL_STATUS,

    "audit_type": (
        "gauge_unfixed_spherical_variation_vs_"
        "gauge_fixed_reduced_radial_variation"
    ),

    "candidate_action_status": "CONDITIONAL_ANSATZ",

    "A_equation_match": True,
    "B_equation_match": True,

    "additional_angular_equation": True,

    "angular_equation_status": (
        "CONSISTENT_THROUGH_SOLVED_WEAK_FIELD_ORDER"
    ),

    "weak_field_rank": int(rank_consistency),

    "rank3_survives_consistency_audit": True,

    "candidate_branch_ppn": {
        "gamma": 1,
        "beta": 1,
        "status": "SPHERICALLY_CONSISTENCY_AUDITED",
    },

    "c14_fixed_by_GVH": False,
    "unique_GVH_action_selected": False,

    "fully_general_4D_tensor_CAS_variation_completed": False,

    "observational_data_used": False,

    "unique_GVH_prediction_ready": False,

    "next_theory_gate": (
        "coupling_selection_stability_or_"
        "fully_general_tensor_crosscheck"
    ),
}

if Path("/content").exists():
    EXPORT_DIR = Path("/content/gvh_exports")
else:
    EXPORT_DIR = Path.cwd() / "gvh_exports"

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

artifact_path = (
    EXPORT_DIR
    / "gvh_0.3.2.6_full_tensor_vs_reduced_radial_consistency_audit.json"
)

artifact_path.write_text(
    json.dumps(
        artifact,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

assert artifact_path.exists()
assert artifact_path.stat().st_size > 0

assert artifact["A_equation_match"] is True
assert artifact["B_equation_match"] is True
assert artifact["rank3_survives_consistency_audit"] is True
assert artifact["observational_data_used"] is False
assert artifact["unique_GVH_prediction_ready"] is False

print("PASS — 0.3.2.6 consistency artifact created.")
print("Artifact:", artifact_path)
print("Status:", FINAL_STATUS)


PASS — 0.3.2.6 consistency artifact created.
Artifact: /content/gvh_exports/gvh_0.3.2.6_full_tensor_vs_reduced_radial_consistency_audit.json
Status: PASS-FULL-SPHERICAL-vs-REDUCED-RADIAL-CONSISTENCY_RANK-3-SURVIVES_PPN-GAMMA-BETA-STRENGTHENED_BLOCKED-FULL-4D-CAS-AND-UNIQUE-GVH-PREDICTION



# Conclusion

0.3.2.6 performs the key consistency check requested after 0.3.2.5.

Starting from the more general spherical metric

\[
ds^2
=
-A\,dt^2
+
B\,dr^2
+
C\,d\Omega^2,
\]

the notebook varies with respect to \(A,B,C\) **before** imposing the areal-radius gauge.

After setting

\[
C=r^2,
\]

the resulting \(A\) and \(B\) equations reproduce the gauge-fixed radial equations of 0.3.2.5 exactly up to nonsingular multiplicative factors.

The additional angular equation is also satisfied through the weak-field orders that were actually solved in 0.3.2.5.

Therefore

\[
\boxed{\operatorname{rank}J=3}
\]

survives the consistency audit, and the conditional aligned-branch results

\[
\boxed{\gamma=1},
\qquad
\boxed{\beta=1}
\]

are strengthened.

However the correct scientific boundary remains explicit:

- \(c_{14}\) is not fixed;
- the candidate action is not uniquely derived from GVH;
- the completely general 4D tensor variation has not yet been independently reproduced;
- no unique non-GR GVH prediction is claimed.

Expected status:

```text
PASS-FULL-SPHERICAL-vs-REDUCED-RADIAL-CONSISTENCY_RANK-3-SURVIVES_PPN-GAMMA-BETA-STRENGTHENED_BLOCKED-FULL-4D-CAS-AND-UNIQUE-GVH-PREDICTION
```
